# RealEstatePRO

In [ ]:
#import all necessary libraries
import pandas as pd
import os
from getpass import getpass
from langchain_openai import OpenAIEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import tqdm
from shapely.geometry import Point, shape
import geopandas as gpd
from pyproj import CRS, Transformer

In [140]:
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY") or \
    getpass("Enter LangSmith API Key: ")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "aurelioai-langchain-course-agent-executor-openai"

In [143]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") \
    or getpass("Enter your OpenAI API key: ")

In [360]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or \
    getpass("Enter Google API Key: ")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "realestatepro-475107-9bec18ee71b8.json"


## DataFrame import

In [354]:
df = pd.read_csv('NY-House-Dataset.csv')

In [355]:
df.head(10)

,BROKERTITLE,TYPE,PRICE,BEDS,BATH,PROPERTYSQFT,ADDRESS,STATE,MAIN_ADDRESS,ADMINISTRATIVE_AREA_LEVEL_2,LOCALITY,SUBLOCALITY,STREET_NAME,LONG_NAME,FORMATTED_ADDRESS,LATITUDE,LONGITUDE
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.000000,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.000000,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.000000,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.000000,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.000000,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856
5,Brokered by Sowae Corp,House for sale,690000,5,2.000000,4004.000000,584 Park Pl,"Brooklyn, NY 11238","584 Park PlBrooklyn, NY 11238",United States,New York,Kings County,Brooklyn,Park Place,"584 Park Pl, Brooklyn, NY 11238, USA",40.674363,-73.958725
6,Brokered by Douglas Elliman - 575 Madison Ave,Condo for sale,899500,2,2.000000,2184.207862,157 W 126th St Unit 1B,"New York, NY 10027","157 W 126th St Unit 1BNew York, NY 10027",New York,New York County,New York,Manhattan,157,"157 W 126th St #1b, New York, NY 10027, USA",40.809448,-73.946777
7,Brokered by Connie Profaci Realty,House for sale,16800000,8,16.000000,33000.000000,177 Benedict Rd,"Staten Island, NY 10304","177 Benedict RdStaten Island, NY 10304",United States,New York,Richmond County,Staten Island,Benedict Road,"177 Benedict Rd, Staten Island, NY 10304, USA",40.595002,-74.106424
8,Brokered by Pantiga Group Inc.,Co-op for sale,265000,1,1.000000,750.000000,875 Morrison Ave Apt 3M,"Bronx, NY 10473","875 Morrison Ave Apt 3MBronx, NY 10473",Bronx County,The Bronx,East Bronx,Morrison Avenue,Parking lot,"Parking lot, 875 Morrison Ave #3m, Bronx, NY 1...",40.821586,-73.874089
9,Brokered by CENTURY 21 MK Realty,Co-op for sale,440000,2,1.000000,978.000000,1350 Ocean Pkwy Apt 5G,"Brooklyn, NY 11230","1350 Ocean Pkwy Apt 5GBrooklyn, NY 11230",New York,Kings County,Brooklyn,Midwood,1350,"1350 Ocean Pkwy #5g, Brooklyn, NY 11230, USA",40.615738,-73.969694


In [356]:
df.describe()

,PRICE,BEDS,BATH,PROPERTYSQFT,LATITUDE,LONGITUDE
count,4.801000e+03,4801.000000,4801.000000,4801.000000,4801.000000,4801.000000
mean,2.356940e+06,3.356801,2.373861,2184.207862,40.714227,-73.941601
std,3.135525e+07,2.602315,1.946962,2377.140894,0.087676,0.101082
min,2.494000e+03,1.000000,0.000000,230.000000,40.499546,-74.253033
25%,4.990000e+05,2.000000,1.000000,1200.000000,40.639375,-73.987143
50%,8.250000e+05,3.000000,2.000000,2184.207862,40.726749,-73.949189
75%,1.495000e+06,4.000000,3.000000,2184.207862,40.771923,-73.870638
max,2.147484e+09,50.000000,50.000000,65535.000000,40.912729,-73.702450


In [146]:
df.shape

(4801, 17)

In [357]:
#make all columns lowercase
df.columns = [col.lower() for col in df.columns]

In [358]:
df.columns

Index(['brokertitle', 'type', 'price', 'beds', 'bath', 'propertysqft',
       'address', 'state', 'main_address', 'administrative_area_level_2',
       'locality', 'sublocality', 'street_name', 'long_name',
       'formatted_address', 'latitude', 'longitude'],
      dtype='object')

### Save embeddings in FAISS

What is FAISS: TODO

In [398]:
index_path = "faiss_index_dir"
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
google_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [368]:
if os.path.exists(index_path):
    # Load existing FAISS index
    vector_store = FAISS.load_local(index_path, google_embeddings, allow_dangerous_deserialization=True)
    print("Loaded existing FAISS index.")

Loaded existing FAISS index.


In [399]:
# Create a concise textual representation for embedding
df['text_to_embed'] = (
    df['brokertitle'].astype(str) + ", " +
    df['type'].astype(str) + ", Price: " + df['price'].astype(str) + "$, " +
    "Beds: " + df['beds'].astype(str) + ", Baths: " + df['bath'].astype(str) + ", " +
    "Size: " + df['propertysqft'].astype(str) + " sqft, " +
    "Address: " + df['address'].astype(str) + ", " +
    "Locality: " + df['locality'].astype(str) + ", " +
    "State: " + df['state'].astype(str)
)

In [374]:
df

,brokertitle,type,price,beds,bath,propertysqft,address,state,main_address,administrative_area_level_2,locality,sublocality,street_name,long_name,formatted_address,latitude,longitude,text_to_embed
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.000000,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483,"Brokered by Douglas Elliman -111 Fifth Ave, C..."
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.000000,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991,"Brokered by Serhant, Condo for sale, Price: 19..."
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.000000,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109,"Brokered by Sowae Corp, House for sale, Price:..."
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.000000,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613,"Brokered by COMPASS, Condo for sale, Price: 69..."
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.000000,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856,Brokered by Sotheby's International Realty - E...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4796,Brokered by COMPASS,Co-op for sale,599000,1,1.000000,2184.207862,222 E 80th St Apt 3A,"Manhattan, NY 10075","222 E 80th St Apt 3AManhattan, NY 10075",New York,New York County,New York,Manhattan,222,"222 E 80th St #3a, New York, NY 10075, USA",40.774350,-73.955879,"Brokered by COMPASS, Co-op for sale, Price: 59..."
4797,Brokered by Mjr Real Estate Llc,Co-op for sale,245000,1,1.000000,2184.207862,97-40 62 Dr Unit Lg,"Rego Park, NY 11374","97-40 62 Dr Unit LgRego Park, NY 11374",United States,New York,Queens County,Queens,62nd Drive,"97-40 62nd Dr, Rego Park, NY 11374, USA",40.732538,-73.860152,"Brokered by Mjr Real Estate Llc, Co-op for sal..."
4798,Brokered by Douglas Elliman - 575 Madison Ave,Co-op for sale,1275000,1,1.000000,2184.207862,427 W 21st St Unit Garden,"New York, NY 10011","427 W 21st St Unit GardenNew York, NY 10011",United States,New York,New York County,New York,West 21st Street,"427 W 21st St, New York, NY 10011, USA",40.745882,-74.003398,"Brokered by Douglas Elliman - 575 Madison Ave,..."
4799,Brokered by E Realty International Corp,Condo for sale,598125,2,1.000000,655.000000,91-23 Corona Ave Unit 4G,"Elmhurst, NY 11373","91-23 Corona Ave Unit 4GElmhurst, NY 11373",New York,Queens County,Queens,Flushing,91-23,"91-23 Corona Ave. #4b, Flushing, NY 11373, USA",40.742770,-73.872752,"Brokered by E Realty International Corp, Condo..."


In [402]:
# Create LangChain Documents, saving only essential metadata
documents = []
for _, row in df.iterrows():
    metadata = {
        "latitude": float(row['latitude']),
        "longitude": float(row['longitude']),
        "address": row['address'],
        "property_id": row.get('property_id', None)  # if you have unique IDs
    }
    doc = Document(page_content=row['text_to_embed'], metadata=metadata)
    documents.append(doc)

In [403]:
embedding_dim = len(google_embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=google_embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [404]:
from tqdm import tqdm

for doc in tqdm(documents):
    vector_store.add_documents([doc])

100%|██████████| 4801/4801 [19:34<00:00,  4.09it/s]


In [187]:
results = vector_store.similarity_search(
    "House in New York with 3 bedrooms and not more than 2 bathrooms between 1000000$ and 3000000$",
    k=2
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Brokered by COMPASS, House for sale, Price: 14000000$, Beds: 3, Baths: 2.3738608579684373, Size: 23027.0 sqft, Address: 39 Eldridge St, Locality: New York, State: Manhattan, NY 10002, Latitude: 40.7157769, Longitude: -73.99357 [{'beds': 3, 'bath': 2.3738608579684373, 'address': '39 Eldridge St', 'state': 'Manhattan, NY 10002', 'propertysqft': 23027.0, 'price': 14000000, 'latitude': 40.7157769, 'longitude': -73.99357}]
* Brokered by COMPASS, Condo for sale, Price: 3000000$, Beds: 3, Baths: 3.0, Size: 2100.0 sqft, Address: 418 E 59th St # B, Locality: New York, State: Manhattan, NY 10022, Latitude: 40.7590132, Longitude: -73.9612618 [{'beds': 3, 'bath': 3.0, 'address': '418 E 59th St # B', 'state': 'Manhattan, NY 10022', 'propertysqft': 2100.0, 'price': 3000000, 'latitude': 40.7590132, 'longitude': -73.9612618}]


In [405]:
vector_store.save_local("faiss_index_dir")

## Park data

In [384]:
df_park = pd.read_csv('Parks_Properties_20251011.csv')

In [ ]:
index_path = "faiss_index_dir_parks"
if os.path.exists(index_path):
    # Load existing FAISS index
    vector_store_parks = FAISS.load_local(index_path, google_embeddings, allow_dangerous_deserialization=True)
    print("Loaded existing FAISS index.")

Loaded existing FAISS index.


In [385]:
#lowercase all columns
df_park.columns = [col.lower() for col in df_park.columns]
df_park['multipolygon'] = df_park['multipolygon'].apply(wkt.loads)

In [386]:
df_park

,acquisitiondate,acres,address,borough,class,communityboard,councildistrict,department,eapply,gisobjid,...,pip_ratable,precinct,retired,signname,subcategory,typecategory,us_congress,waterfront,zipcode,multipolygon
0,04/19/1938 12:00:00 AM,1.837,88-02 ATLANTIC AVENUE,Q,PARK,409,32,Q-09,London Planetree Playground,100000165.0,...,True,102.0,False,London Planetree Playground,Neighborhood Plgd,Neighborhood Park,7.0,False,11416,MULTIPOLYGON (((-73.85277880723194 40.68568340...
1,06/27/1934 12:00:00 AM,19.749,3324 RESERVOIR OVAL EAST,X,PARK,207,11,X-07,Williamsbridge Oval,100003776.0,...,True,52.0,False,Williamsbridge Oval,Large Park,Neighborhood Park,15.0,False,10467,MULTIPOLYGON (((-73.87770412084713 40.87616201...
2,NaN,0.137,NaN,Q,PARK,410,32,Q-10,Tudor Malls,100000448.0,...,True,106.0,False,Tudor Malls,Sitting Area/Triangle/Mall,Mall,57.0,False,11417,MULTIPOLYGON (((-73.85048341833254 40.67426760...
3,08/02/1972 12:00:00 AM,1.578,NaN,M,PARK,107,6,M-14,Joan Of Arc Park,100004772.0,...,False,24.0,False,Joan Of Arc Park,Neighborhood Park,Neighborhood Park,12.0,False,"10024, 10025",MULTIPOLYGON (((-73.9768661390178 40.792971431...
4,09/22/1900 12:00:00 AM,13.350,675 RIVERSIDE DRIVE,M,PARK,109,7,M-14,Riverside Park,100005045.0,...,False,30.0,False,Riverside Park,Flagship Park,Neighborhood Park,13.0,False,"10027, 10031",MULTIPOLYGON (((-73.95194430353968 40.82760315...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2049,04/09/1908 12:00:00 AM,1.300,NaN,M,PARK,109,7,M-09,Broadway Malls 110th-122nd,100005116.0,...,False,26.0,False,Broadway Malls,Sitting Area/Triangle/Mall,Mall,1213.0,False,"10025, 10027",MULTIPOLYGON (((-73.9638091050991 40.808084800...
2050,01/25/1945 12:00:00 AM,3.253,NaN,B,PARK,316,37,B-16,Callahan-Kelly Playground,100004353.0,...,True,73.0,False,Callahan-Kelly Playground,Neighborhood Plgd,Community Park,78.0,False,11233,MULTIPOLYGON (((-73.90369797331626 40.67860334...
2051,08/13/1947 12:00:00 AM,6.300,180 SANDS STREET,B,PARK,302,33,B-02,Trinity Park,100003928.0,...,True,84.0,False,Trinity Park,Sitting Area/Triangle/Mall,Parkway,7.0,False,11201,MULTIPOLYGON (((-73.98333289232065 40.69848030...
2052,04/20/1938 12:00:00 AM,170.700,1 WARDS ISLAND,M,PARK,111,8,M-11R,Wards Island Park,100003801.0,...,False,25.0,False,Wards Island Park,Flagship Park,Recreational Field/Courts,13.0,True,10035,MULTIPOLYGON (((-73.9316174582867 40.783628272...


In [387]:
#select only relevant columns
df_park = df_park[['eapply',  'signname', 'subcategory', 'typecategory',  'multipolygon']]

In [388]:
df_park

,eapply,signname,subcategory,typecategory,multipolygon
0,London Planetree Playground,London Planetree Playground,Neighborhood Plgd,Neighborhood Park,MULTIPOLYGON (((-73.85277880723194 40.68568340...
1,Williamsbridge Oval,Williamsbridge Oval,Large Park,Neighborhood Park,MULTIPOLYGON (((-73.87770412084713 40.87616201...
2,Tudor Malls,Tudor Malls,Sitting Area/Triangle/Mall,Mall,MULTIPOLYGON (((-73.85048341833254 40.67426760...
3,Joan Of Arc Park,Joan Of Arc Park,Neighborhood Park,Neighborhood Park,MULTIPOLYGON (((-73.9768661390178 40.792971431...
4,Riverside Park,Riverside Park,Flagship Park,Neighborhood Park,MULTIPOLYGON (((-73.95194430353968 40.82760315...
...,...,...,...,...,...
2049,Broadway Malls 110th-122nd,Broadway Malls,Sitting Area/Triangle/Mall,Mall,MULTIPOLYGON (((-73.9638091050991 40.808084800...
2050,Callahan-Kelly Playground,Callahan-Kelly Playground,Neighborhood Plgd,Community Park,MULTIPOLYGON (((-73.90369797331626 40.67860334...
2051,Trinity Park,Trinity Park,Sitting Area/Triangle/Mall,Parkway,MULTIPOLYGON (((-73.98333289232065 40.69848030...
2052,Wards Island Park,Wards Island Park,Flagship Park,Recreational Field/Courts,MULTIPOLYGON (((-73.9316174582867 40.783628272...


In [389]:
#remove all rows with NaN
df_park = df_park.dropna()


In [390]:
# Create a concise textual representation for embedding
df_park['text_to_embed'] = (
    df_park['eapply'].astype(str) + ", Signname: " + df_park['signname'].astype(str) + ", Subcategory: " + df_park['subcategory'].astype(str) + ", Typecategory: " + df_park['typecategory'].astype(str) + ", Location: " + df_park['multipolygon'].astype(str)
)

C:\Users\gianl_2\AppData\Local\Temp\ipykernel_6284\2992300739.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_park['text_to_embed'] = (


In [391]:
# Create LangChain Documents, saving only essential metadata
park_documents = []
for idx, row in df_park.iterrows():
    metadata = {
        "eapply": row['eapply'],
        "signname": row['signname'],
        "subcategory": row['subcategory'],
        "typecategory": row['typecategory'],
        "multipolygon": row['multipolygon']
    }
    doc = Document(page_content=row['text_to_embed'], metadata=metadata)
    park_documents.append(doc)

In [392]:
embedding_dim = len(google_embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store_parks = FAISS(
    embedding_function=google_embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [396]:
from tqdm import tqdm

for doc in tqdm(park_documents):
    vector_store_parks.add_documents([doc])


 87%|████████▋ | 1617/1869 [06:57<01:05,  3.88it/s]


GoogleGenerativeAIError: Error embedding content: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
]

In [260]:
results = vector_store_parks.similarity_search(
    "Park called Central Park",
    k=2
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* First Park, Address: 2 2 AVENUEMULTIPOLYGON (((-73.9904178720097 40.72401752715714, -73.99051656153274 40.72382819425332, -73.99055231835985 40.72375963151698, -73.99085066541139 40.723964516063916, -73.99083684679613 40.723780765151076, -73.98874205680221 40.72314934757435, -73.98865201601932 40.72327357930694, -73.98961617206143 40.72367978018234, -73.98970382860819 40.72351171790102, -73.9898365244603 40.72355182846928, -73.98990536625168 40.72357263719071, -73.98997420926946 40.72359344767201, -73.99004419330399 40.72361460220531, -73.99012866562788 40.72364013621519, -73.99020924165647 40.723664493766655, -73.99029419582615 40.723690172677905, -73.99029950237012 40.72367999737023, -73.9903881342348 40.72370513441529, -73.99038021824195 40.72372031544005, -73.9904686582889 40.723738242029235, -73.9903400912778 40.72398475923113, -73.9904178720097 40.72401752715714))) [{}]
* The Big Park, Address: 110 CONTINENTAL PLACEMULTIPOLYGON (((-74.16375018086438 40.6309532329797, -74.164356

In [161]:
vector_store_parks.save_local("faiss_index_dir_parks")

## LangChain

### LLM

In [397]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="gemini-2.5-pro",
    temperature=0.0,
    streaming=True
)

### Tools

In [ ]:
import re
from langchain_core.tools import tool

@tool
def retrieve(query: str, k: int) -> dict:
    """Retrieve relevant real estate listings based on the query."""
    results = vector_store.similarity_search(query, k=k)
    if not results:
        return "No relevant listings found."
    
    response = "Here are some relevant listings:\n"
    for i, res in enumerate(results, 1):
        response += (f"{i}. {res.page_content} | "
                     f"Beds: {res.metadata.get('beds')}, "
                     f"Baths: {res.metadata.get('bath')}, "
                     f"Price: {res.metadata.get('price')}$, "
                     f"Address: {res.metadata.get('address')}, "
                     f"State: {res.metadata.get('state')}, "
                     f"Size: {res.metadata.get('propertysqft')} sqft, "
                     f"Latitude: {res.metadata.get('latitude')}, "
                     f"Longitude: {res.metadata.get('longitude')}\n")
    return {"answer": response, "tools_used": ["retrieve"]}

@tool
def average_price(location: str) -> dict:
    """Calculate the average price of houses in a given location."""
    results = vector_store.similarity_search(f"houses in {location}", k=20)
    if not results:
        return {"answer": f"No listings found for location: {location}", "tools_used": ["average_price"]}
    prices = [res.metadata.get('price') for res in results if res.metadata.get('price') is not None]
    if not prices:
        return {"answer": f"No price data available for listings in {location}", "tools_used": ["average_price"]}
    
    avg_price = sum(prices) / len(prices)
    return {"answer": f"The average price of houses in {location} is approximately ${avg_price:,.2f}.", "tools_used": ["average_price"]}

@tool
def retrieve_parks(query: str) -> dict:
    """Retrieve relevant parks based on the query."""
    results = vector_store_parks.similarity_search(query, k=50)
    if not results:
        return "No relevant parks found."
    
    response = "Here are some relevant parks:\n"
    for i, res in enumerate(results, 1):
        response += (f"{i}. {res.page_content} | "
                     f"{res.metadata.get('eapply')}\n")
    return {"answer": response, "tools_used": ["retrieve_parks"]}

def find_nearest_park(house_lat, house_lon, df_park):
    # Convert 'multipolygon' column from WKT strings to shapely MultiPolygon objects
    parks_gdf = gpd.GeoDataFrame(df_park, geometry='multipolygon', crs='EPSG:4326')
    parks_gdf = parks_gdf.to_crs(epsg=3857)
    house_point = gpd.GeoSeries([Point(house_lon, house_lat)], crs='EPSG:4326').to_crs(epsg=3857).iloc[0]
    parks_gdf = parks_gdf.to_crs(epsg=3857)
    parks_gdf['distance'] = parks_gdf.geometry.distance(house_point)
    nearest_park_row = parks_gdf.loc[parks_gdf['distance'].idxmin()]
    distance_km = nearest_park_row['distance'] / 1000  # meters to km
    return nearest_park_row, distance_km    

@tool
def find_nearest_park_to_house(house_response: str) -> dict:
    """
    Input: response string from 'retrieve' tool with house listings.
    Parses house lat/lon from the response, finds nearest park for each house, 
    returns formatted textual results. Use only if relevant parks are requested.
    """
    # Regex to extract lat/lon pairs from the house response string
    coords = re.findall(r"Latitude:\s*([-+]?\d*\.\d+|\d+),\s*Longitude:\s*([-+]?\d*\.\d+|\d+)", house_response)
    if not coords:
        return {"answer": "No valid house coordinates found in the input.", "tools_used": ["find_nearest_park_to_house"]}

    results = []
    for i, (lat_str, lon_str) in enumerate(coords, 1):
        lat, lon = float(lat_str), float(lon_str)
        nearest_park, dist_km = find_nearest_park(lat, lon, df_park)
        if nearest_park is None:
            results.append(f"House {i}: No parks found.")
        else:
            results.append(f"House {i}: The nearest park is '{nearest_park['signname']}' approx. {dist_km:.3f} km away.")
    
    return {"answer": "\n".join(results), "tools_used": ["find_nearest_park_to_house"]}



@tool
def final_answer(answer: str, tools_used: list[str]) -> str:
    """Use this tool to provide a final answer to the user.
    The answer should be in natural language as this will be provided
    to the user directly. The tools_used must include a list of tool
    names that were used within the `scratchpad`.
    """
    return {"answer": answer, "tools_used": tools_used}

### Creating Agent

### Chat Prompt Template

In [332]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI assistant helping users find real estate listings and nearby parks based on their queries."
     " Use the available tools, when required, to retrieve information and provide accurate answers."
     Try to always reply with a final answer."""),
    
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])


### Agent

In [333]:
from langchain_core.runnables.base import RunnableSerializable

tools = [retrieve, average_price, final_answer, retrieve_parks, find_nearest_park_to_house]

# define the agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")
)

In [334]:
import asyncio
from langchain.callbacks.base import AsyncCallbackHandler


class QueueCallbackHandler(AsyncCallbackHandler):
    """Callback handler that puts tokens into a queue."""

    def __init__(self, queue: asyncio.Queue):
        self.queue = queue
        self.final_answer_seen = False

    async def __aiter__(self): #necessary for any async method
        while True:
            if self.queue.empty():
                await asyncio.sleep(0.1) #how quickly token is added to the queue
                continue
            token_or_done = await self.queue.get()

            if token_or_done == "<<DONE>>":
                # this means we're done
                return
            if token_or_done:
                yield token_or_done #return a token but continue the loop generator

    async def on_llm_new_token(self, *args, **kwargs) -> None:
        """Put new token in the queue."""
        #print(f"on_llm_new_token: {args}, {kwargs}")
        chunk = kwargs.get("chunk")
        if chunk:
            # check for final_answer tool call
            if tool_calls := chunk.message.additional_kwargs.get("tool_calls"):
                if tool_calls[0]["function"]["name"] == "final_answer":
                    # this will allow the stream to end on the next `on_llm_end` call
                    self.final_answer_seen = True
        self.queue.put_nowait(kwargs.get("chunk"))
        return

    async def on_llm_end(self, *args, **kwargs) -> None:
        """Put None in the queue to signal completion."""
        #print(f"on_llm_end: {args}, {kwargs}")
        # this should only be used at the end of our agent execution, however LangChain
        # will call this at the end of every tool call, not just the final tool call
        # so we must only send the "done" signal if we have already seen the final_answer
        # tool call
        if self.final_answer_seen:
            self.queue.put_nowait("<<DONE>>")
        else:
            self.queue.put_nowait("<<STEP_END>>")
        return

In [335]:
import json
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage,ToolMessage

# create tool name to function mapping
name2tool = {tool.name: tool.func for tool in tools}

class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3):
        self.chat_history = []
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
        )

    async def invoke(self, input: str, streamer: QueueCallbackHandler, verbose: bool = False) -> dict:
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0
        agent_scratchpad = []
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            async def stream(query: str):
                response = self.agent.with_config(
                    callbacks=[streamer]
                )
                # we initialize the output dictionary that we will be populating with
                # our streamed output
                output = None
                # now we begin streaming
                async for token in response.astream({
                    "input": query,
                    "chat_history": self.chat_history,
                    "agent_scratchpad": agent_scratchpad
                }):
                    if output is None:
                        output = token
                    else:
                        # we can just add the tokens together as they are streamed and
                        # we'll have the full response object at the end
                        output += token
                    if token.content != "":
                        # we can capture various parts of the response object
                        if verbose: print(f"content: {token.content}", flush=True)
                    tool_calls = token.additional_kwargs.get("tool_calls")
                    if tool_calls:
                        if verbose: print(f"tool_calls: {tool_calls}", flush=True)
                        tool_name = tool_calls[0]["function"]["name"]
                        if tool_name:
                            if verbose: print(f"tool_name: {tool_name}", flush=True)
                        arg = tool_calls[0]["function"]["arguments"]
                        if arg != "":
                            if verbose: print(f"arg: {arg}", flush=True)
                return AIMessage(
                    content=output.content,
                    tool_calls=output.tool_calls,
                    tool_call_id=output.tool_calls[0]["id"]
                )

            tool_call = await stream(query=input)
            # add initial tool call to scratchpad
            agent_scratchpad.append(tool_call)
            # otherwise we execute the tool and add it's output to the agent scratchpad
            tool_name = tool_call.tool_calls[0]["name"]
            tool_args = tool_call.tool_calls[0]["args"]
            tool_call_id = tool_call.tool_call_id
            tool_out = name2tool[tool_name](**tool_args)
            # add the tool output to the agent scratchpad
            tool_exec = ToolMessage(
                content=f"{tool_out}",
                tool_call_id=tool_call_id
            )
            agent_scratchpad.append(tool_exec)
            count += 1
            # if the tool call is the final answer tool, we stop
            if tool_name == "final_answer":
                break
        # add the final output to the chat history, we only add the "answer" field
        final_answer = tool_out["answer"]
        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer)
        ])
        # return the final answer in dict form
        return tool_args

agent_executor = CustomAgentExecutor()

In [339]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

task = asyncio.create_task(agent_executor.invoke("How far is this house from the Empire State Building?", streamer))

async for token in streamer:
    # first identify if we have a <<STEP_END>> token
    if token == "<<STEP_END>>":
        print("\n", flush=True)
    # we'll first identify if the token is a tool call
    elif tool_calls := token.message.additional_kwargs.get("tool_calls"):
        # if we have a tool call with a tool name, we'll print it
        if tool_name := tool_calls[0]["function"]["name"]:
            print(f"Calling {tool_name}...", flush=True)
        # if we have a tool call with arguments, we ad them to our args string
        if tool_args := tool_calls[0]["function"]["arguments"]:
            print(f"{tool_args}", end="", flush=True)

_ = await task

Calling retrieve...
{"query":"Empire State Building, New York, NY","k":1}

Calling final_answer...
{"answer":"The house at 135 E 15th St, New York, NY 10003, is approximately 1.6 miles (about 2.6 kilometers) from the Empire State Building. This makes it a short drive or a pleasant walk away from one of New York City's most iconic landmarks.","tools_used":["retrieve"]}